# gguf-serve — any GGUF model, one public OpenAI-compatible API

Run the two cells below. The first clones the repo, the second does everything else:
installs the CUDA build of llama.cpp, downloads and verifies the model, loads it across
your GPUs, and serves a chat UI plus an OpenAI-compatible API on one public URL.

**Before you start**, set the accelerator to **GPU T4 x2**
(Kaggle: *Settings -> Accelerator*). The default model, Qwen3.8-27B at Q5_K_XL, needs
about 23 GiB of VRAM — that is two T4s.

On free Colab you only get a single 16 GB T4 and the default will not fit. Use an L4 or
A100 runtime, or serve a smaller model by changing the launch command in cell 2 to:

```
!python launch.py --model-file Qwen3.8-27B-UD-Q3_K_XL.gguf
```

First run takes roughly 15 minutes, most of it downloading the model.

In [ ]:
# 1 — Get the code
#
# Change REPO_URL if you forked the repo.

REPO_URL = "https://github.com/kishorsharma/gguf-serve.git"

import os
from pathlib import Path

workdir = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/content")
repo = workdir / "gguf-serve"

if repo.exists():
    print(f"{repo} already exists — pulling the latest changes")
    !cd {repo} && git pull --ff-only
else:
    !git clone --depth 1 {REPO_URL} {repo}

os.chdir(repo)
print("\nworking directory:", Path.cwd())

In [ ]:
# 2 — Install, download, load, serve
#
# Leave this cell running. The server lives inside it, so stopping the cell
# stops the server and kills the public URL.
#
# Watch for the https://....gradio.live line in the output — that is your
# public URL. Open it for the chat UI, add /docs for the API reference,
# or point any OpenAI client at <url>/v1.
#
# To serve a different model, add:
#   --model-repo <hf-repo> --model-file <file.gguf>

!python launch.py

## Using the API from anywhere

Once the public URL is up, any OpenAI client works against it. Run this from your
laptop, not from this notebook:

```python
from openai import OpenAI

client = OpenAI(
    base_url="https://YOUR-ID.gradio.live/v1",
    api_key="not-used",  # this server does not check keys
)

response = client.chat.completions.create(
    model="qwen3.8-27b-ud-q5-k-xl",
    messages=[{"role": "user", "content": "Hello!"}],
)

print(response.choices[0].message.content)
```

`GET /health` reports the exact model id if you changed the model. The URL is
temporary: it is gone as soon as this notebook stops.

## Notes

- **The model is downloaded to `/tmp` and lost on restart.** `/kaggle/working` is
  capped at 20 GB, which is too small. To keep it, add the GGUF as a Kaggle dataset
  and pass `--model-dir /kaggle/input/<your-dataset>`.
- **Anyone with the public link can use the model.** There is no authentication.
  Use `--no-share` if you only want local access.
- To change the context size, GPU split, or sampling defaults, edit
  `ggufserve/config.py`. See `docs/configuration.md`.